> Adapted from *Dive into Deep Learning*, section [File I/O](https://d2l.ai/chapter_builders-guide/read-write.html). Not affiliated with or endorsed by the book's authors.
> Source, authors, licenses and changes: [Attribution and changes](#Attribution-and-changes), at the end of this notebook.

# File I/O

So far we have discussed how to process data and how
to build, train, and test deep learning models.
However, at some point we will hopefully be happy enough
with the learned models that we will want
to save the results for later use in various contexts
(perhaps even to make predictions in deployment).
Additionally, when running a long training process,
the best practice is to periodically save intermediate results (checkpointing)
to ensure that we do not lose several days' worth of computation
if we trip over the power cord of our server.
Thus it is time to learn how to load and store
both individual weight vectors and entire models.
This section addresses both issues.

In [ ]:
import numpy as np
from divelet_ml import nn

import divelet_ml as d2l

## Loading and Saving Tensors

For individual tensors, we can directly
invoke NumPy's `save` and `load` functions
to read and write them respectively.
Both functions require that we supply a name,
and `save` requires as input the variable to be saved.
`save` adds the extension `.npy` to the name.

In [ ]:
x = np.arange(4)
np.save('x-file', x)

We can now read the data from the stored file back into memory.

In [ ]:
x2 = np.load('x-file.npy')
x2

We can store several tensors in one file with `savez`, which adds `.npz`, and read them back into memory.

In [ ]:
y = np.zeros(4)
np.savez('x-files', x, y)
files = np.load('x-files.npz')
x2, y2 = files['arr_0'], files['arr_1']
(x2, y2)

We can even [**write and read a dictionary that maps
from strings to tensors.**]
This is convenient when we want
to read or write all the weights in a model.

In [ ]:
mydict = {'x': x, 'y': y}
np.savez('mydict', **mydict)
mydict2 = dict(np.load('mydict.npz'))
mydict2

## Loading and Saving Model Parameters

Saving individual weight vectors (or other tensors) is useful,
but it gets very tedious if we want to save
(and later load) an entire model.
After all, we might have hundreds of
parameter groups sprinkled throughout.
For this reason the deep learning framework provides built-in functionalities
to load and save entire networks.
An important detail to note is that this
saves model *parameters* and not the entire model.
For example, if we have a 3-layer MLP,
we need to specify the architecture separately.
The reason for this is that the models themselves can contain arbitrary code,
hence they cannot be serialized as naturally.
Thus, in order to reinstate a model, we need
to generate the architecture in code
and then load the parameters from disk.
Let's start with our familiar MLP.

In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.LazyLinear(256)
        self.output = nn.LazyLinear(10)

    def forward(self, x):
        return self.output(nn.relu(self.hidden(x)))

net = MLP()
X = np.random.randn(2, 20)
Y = net(X)

Next, we store the parameters of the model as a file with the name "mlp.params.npz".

In [ ]:
np.savez('mlp.params', **net.state_dict())

To recover the model, we instantiate a clone
of the original MLP model.
Instead of randomly initializing the model parameters,
we read the parameters stored in the file directly.

In [ ]:
clone = MLP()
clone.load_state_dict(np.load('mlp.params.npz'))
clone.eval()

Since both instances have the same model parameters,
the computational result of the same input `X` should be the same.
Let's verify this.

In [ ]:
Y_clone = clone(X)
Y_clone == Y

## Summary

The `save`, `savez` and `load` functions can be used to perform file I/O for tensor objects.
We can save and load the entire sets of parameters for a network via a parameter dictionary.
Saving the architecture has to be done in code rather than in parameters.

## Exercises

1. Even if there is no need to deploy trained models to a different device, what are the practical benefits of storing model parameters?
1. Assume that we want to reuse only parts of a network to be incorporated into a network having a different architecture. How would you go about using, say the first two layers from a previous network in a new network?
1. How would you go about saving the network architecture and parameters? What restrictions would you impose on the architecture?



Discussions of the original section: [book's forum](https://discuss.d2l.ai/t/61)

## Attribution and changes

Adapted from *Dive into Deep Learning*, section
[File I/O](https://d2l.ai/chapter_builders-guide/read-write.html)
(source: [`chapter_builders-guide/read-write.md`](https://github.com/d2l-ai/d2l-en/blob/23d7a5aecceee57d1292c56e90cce307f183bb0a/chapter_builders-guide/read-write.md)
at commit `23d7a5aeccee`), by Aston Zhang, Zachary C. Lipton, Mu Li, Alexander J. Smola and contributors.
Text and figures: [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/). Code: adapted from the book's sample code, MIT-0.
Not affiliated with or endorsed by the book's authors.

**Changes from the original** (PyTorch edition):
- Code uses NumPy instead of PyTorch, so it runs in JupyterLite (Pyodide and xeus-python). The word *tensor* is kept and means a NumPy `ndarray`.
- `d2l` is divelet-ml's package (`import divelet_ml as d2l`) instead of the book's `d2l.torch`.
- `torch.save` and `torch.load` are NumPy's `np.save`, `np.savez` and `np.load`, which add the extensions `.npy` and `.npz` to the file names; a list of tensors is saved with `savez` and read back by position (`arr_0`, `arr_1`).
- A model's parameters are saved with `net.state_dict()` and restored with `load_state_dict`, as in PyTorch. In JupyterLite, the files are written to the kernel's file system, next to the notebook.
- `torch.arange`, `torch.zeros` and `torch.randn` are `np.arange`, `np.zeros` and `np.random.randn`; `F.relu` is `nn.relu`.
- Cross-references to other sections link to the book's site, and citations are written inline (author and year); the book's bibliography lists them.
- The d2lbook framework tabs for MXNet, TensorFlow and JAX are removed.